In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import warnings
import re
warnings.filterwarnings('ignore')
from datetime import datetime

In [2]:
# ==========================================
# 1. SETUP PATHS
# ==========================================

# ── FULL SCALE PATHS ──────────────────────────────────

# Full Philippines OSM shapefile folder
base_osm_dir = "/Users/ruben/Desktop/Thesis/TrainingData/PH_OSM.shp"

# Full DHS GPS shapefile (all 1247 clusters, same file as before)
dhs_shp_path = "/Users/ruben/Desktop/Thesis/TrainingData/PH_DHS_GPS/PHGE81FL/PHGE81FL.shp"

# Full VIIRS labels CSV (generated from viirs_ntl_labels_all_clusters.csv)
viirs_csv_path = "/Users/ruben/Desktop/Thesis/TrainingData/final-data/viirs_ntl_labels_all_clusters.csv"

# Output: full static features for all 1247 clusters
output_csv = "/Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_expanded.csv"

In [3]:
# ==========================================
# 2. LOAD DATA
# ==========================================
print("Loading Shapefiles...")
gdf_dhs = gpd.read_file(dhs_shp_path)

# Load OSM files (Handle missing files gracefully)
try:
    gdf_roads = gpd.read_file(os.path.join(base_osm_dir, "gis_osm_roads_free_1.shp"))
    gdf_bldgs = gpd.read_file(os.path.join(base_osm_dir, "gis_osm_buildings_a_free_1.shp"))
    gdf_pois = gpd.read_file(os.path.join(base_osm_dir, "gis_osm_pois_free_1.shp"))
    print("Loaded Roads, Buildings, and POIs.")
except Exception as e:
    print(f"Error loading OSM files: {e}")
    print("Ensure you have roads, buildings, and POIs in the folder.")

Loading Shapefiles...
Loaded Roads, Buildings, and POIs.


In [4]:
# ==========================================
# 3. REPROJECT & BUFFER
# ==========================================
# Project to Meters (Philippines Zone 51N)
target_crs = "EPSG:32651"
gdf_dhs = gdf_dhs.to_crs(target_crs)
gdf_roads = gdf_roads.to_crs(target_crs)
gdf_bldgs = gdf_bldgs.to_crs(target_crs)
gdf_pois = gdf_pois.to_crs(target_crs)

# DYNAMIC BUFFERING (Tingzon Method)
# Check if DHS has 'URBAN_RURAL' column (usually 'URBAN_RURA' in shapefiles)
# U = Urban, R = Rural. If missing, default to 5km.
print("Creating Dynamic Buffers...")
def get_buffer(row):
    # Adjust column name if necessary (e.g., 'URBAN_RURA', 'TYPE')
    if 'URBAN_RURA' in row and row['URBAN_RURA'] == 'U':
        return row.geometry.buffer(2000) # 2km for Urban
    else:
        return row.geometry.buffer(5000) # 5km for Rural/Unknown

gdf_dhs['buffer_geom'] = gdf_dhs.apply(get_buffer, axis=1)

Creating Dynamic Buffers...


In [5]:
# ==========================================
# 4. LOAD OPTIONAL LANDUSE LAYER
# ==========================================
# Landuse polygons (gis_osm_landuse_a_free_1.shp) are part of the
# standard Geofabrik Philippines download. If the file is missing,
# all landuse features are set to 0 and a warning is printed.

landuse_path = os.path.join(
    base_osm_dir, 'gis_osm_landuse_a_free_1.shp')

if os.path.exists(landuse_path):
    gdf_landuse  = gpd.read_file(landuse_path).to_crs(target_crs)
    lu_sindex    = gdf_landuse.sindex
    HAS_LANDUSE  = True
    print(f'Landuse layer loaded: {len(gdf_landuse)} polygons')
    print('Landuse classes present:')
    print(gdf_landuse['fclass'].value_counts().head(15).to_string())
else:
    HAS_LANDUSE  = False
    print('WARNING: gis_osm_landuse_a_free_1.shp not found.')
    print('  All LU_* features will be 0.')
    print('  Download from: download.geofabrik.de/asia/philippines.html')

Landuse layer loaded: 257531 polygons
Landuse classes present:
fclass
residential          100320
farmland              70981
forest                21957
orchard               12322
industrial             9017
scrub                  7187
retail                 7170
grass                  6702
commercial             6232
park                   5077
cemetery               4116
farmyard               2290
meadow                 1959
recreation_ground       884
quarry                  572


In [6]:
# ==========================================
# 5. FEATURE ENGINEERING LOOP
# ==========================================
# Changes from original notebook:
#
# BUILDINGS — now extracts 4 metrics per type (count, total area,
#   mean area, proportion of buffer area), matching Tingzon et al.
#   (2019). All geometries are clipped to the buffer once and reused
#   for typed subsets, keeping runtime comparable to the original.
#
# POIs — extended from 6 categories to 14:
#   Added: market, atm, pharmacy, fuel, ferry_terminal,
#          bus_stop/bus_station, place_of_worship, government
#
# LANDUSE — new: residential, commercial, industrial,
#   agricultural, and forest area in m² (requires landuse shapefile).
#
# DERIVED — new: Mean_Bldg_Area (all buildings) and
#   Bldg_Density_per_km2 (count normalised by buffer area).

print('Extracting features (Roads, Buildings, POIs, Landuse)...')
results = []

road_sindex = gdf_roads.sindex
bldg_sindex = gdf_bldgs.sindex
poi_sindex  = gdf_pois.sindex

# ── Road type mapping ──────────────────────────────────────────────
ROAD_TYPES = {
    'Main_Roads'     : ['motorway', 'trunk',
                        'primary', 'primary_link'],
    'Secondary_Roads': ['secondary', 'tertiary'],
    'Local_Roads'    : ['residential', 'living_street',
                        'unclassified', 'service'],
    'Tracks'         : ['track', 'path'],
}

# ── Building type mapping ──────────────────────────────────────────
BLDG_TYPES = {
    'residential': ['residential', 'house',
                    'apartments', 'detached'],
    'commercial' : ['commercial', 'retail',
                    'office', 'supermarket'],
    'industrial' : ['industrial', 'warehouse', 'factory'],
    'school'     : ['school', 'university',
                    'college', 'kindergarten'],
    'hospital'   : ['hospital', 'clinic', 'health_centre'],
}

# ── POI type mapping ────────────────────────────────────────────────
# fclass values verified against gis_osm_pois_free_1.shp (PH 2026).
# Tags not present in the shapefile are excluded to avoid zero columns.
# Dropped from earlier version (0 records in PH shapefile):
#   fuel, ferry_terminal, bus_stop/bus_station, place_of_worship
POI_TYPES = {
    # ── Financial ──────────────────────────────────────────────
    'bank'        : ['bank'],
    'atm'         : ['atm'],
    # ── Accommodation ──────────────────────────────────────────
    'hotel'       : ['hotel', 'motel', 'guesthouse', 'hostel'],
    # ── Food & retail ──────────────────────────────────────────
    'fast_food'   : ['fast_food'],
    'convenience' : ['convenience'],
    'restaurant'  : ['restaurant', 'cafe', 'food_court'],
    'market'      : ['market_place', 'supermarket'],
    # ── Education ──────────────────────────────────────────────
    'school'      : ['school', 'college', 'university',
                     'kindergarten'],
    # ── Health ─────────────────────────────────────────────────
    'hospital'    : ['hospital', 'clinic', 'doctors'],
    'pharmacy'    : ['pharmacy'],
    # ── Civic & government ─────────────────────────────────────
    'government'  : ['town_hall', 'public_building'],
    'police'      : ['police', 'fire_station'],
    'post_office' : ['post_office'],
    'community'   : ['community_centre'],
    # ── Leisure & recreation ───────────────────────────────────
    'sports'      : ['pitch', 'sports_centre', 'swimming_pool'],
}

# ── Landuse class mapping ──────────────────────────────────────────
LU_TYPES = {
    'LU_Residential_m2' : ['residential'],
    'LU_Commercial_m2'  : ['commercial', 'retail'],
    'LU_Industrial_m2'  : ['industrial'],
    'LU_Agricultural_m2': ['farmland', 'orchard',
                            'vineyard', 'allotments'],
    'LU_Forest_m2'      : ['forest', 'wood'],
}

for loop_idx, (idx, row) in enumerate(gdf_dhs.iterrows()):
    cluster_id = row['DHSCLUST']
    buffer     = row['buffer_geom']

    # Buffer area in m² for proportion and density calculations.
    # Urban buffer = pi * 2000^2, Rural buffer = pi * 5000^2.
    is_urban      = ('URBAN_RURA' in row
                     and row['URBAN_RURA'] == 'U')
    buffer_area_m2 = np.pi * (2000**2 if is_urban else 5000**2)
    buffer_area_km2 = buffer_area_m2 / 1e6

    rec = {'DHSCLUST': cluster_id}

    # ── ROADS ──────────────────────────────────────────────────────
    possible_roads = gdf_roads.iloc[
        list(road_sindex.intersection(buffer.bounds))]
    precise_roads  = possible_roads[
        possible_roads.intersects(buffer)]

    if len(precise_roads) > 0:
        clipped_roads = precise_roads.geometry.intersection(buffer)
        rec['Total_Road_Length'] = (
            clipped_roads.length.sum() / 1000.0)
        for r_cat, r_classes in ROAD_TYPES.items():
            mask = precise_roads['fclass'].isin(r_classes)
            rec[f'{r_cat}_Length'] = (
                clipped_roads[mask].length.sum() / 1000.0)
    else:
        rec['Total_Road_Length'] = 0.0
        for r_cat in ROAD_TYPES:
            rec[f'{r_cat}_Length'] = 0.0

    # ── BUILDINGS ──────────────────────────────────────────────────
    # Clip ALL buildings once, then filter the clipped GDF by type.
    # This avoids calling geometry.intersection() per type group,
    # keeping runtime similar to the original single-clip approach.
    possible_bldgs = gdf_bldgs.iloc[
        list(bldg_sindex.intersection(buffer.bounds))]
    precise_bldgs  = possible_bldgs[
        possible_bldgs.intersects(buffer)].copy()

    if len(precise_bldgs) > 0:
        # Single clip for all buildings
        precise_bldgs['clip_geom'] = (
            precise_bldgs.geometry.intersection(buffer))
        precise_bldgs['clip_area'] = precise_bldgs['clip_geom'].area

        type_col = ('type' if 'type' in precise_bldgs.columns
                    else 'fclass')

        # Aggregate metrics for ALL buildings combined
        total_count = len(precise_bldgs)
        total_area  = float(precise_bldgs['clip_area'].sum())
        mean_area   = float(precise_bldgs['clip_area'].mean())
        proportion  = round(total_area / buffer_area_m2, 6)

        rec['Total_Bldg_Count']      = total_count
        rec['Total_Bldg_Area']       = round(total_area, 2)
        rec['Mean_Bldg_Area']        = round(mean_area, 2)
        rec['Total_Bldg_Proportion'] = proportion
        rec['Bldg_Density_per_km2']  = round(
            total_count / buffer_area_km2, 4)

        # Per-type metrics: count, total area, mean area, proportion
        for t_name, t_tags in BLDG_TYPES.items():
            subset = precise_bldgs[
                precise_bldgs[type_col].isin(t_tags)]
            t_count = len(subset)
            t_area  = float(subset['clip_area'].sum())
            t_mean  = float(
                subset['clip_area'].mean() if t_count > 0 else 0.0)
            t_prop  = round(t_area / buffer_area_m2, 6)

            rec[f'Bldg_{t_name}_Count']      = t_count
            rec[f'Bldg_{t_name}_TotalArea']  = round(t_area, 2)
            rec[f'Bldg_{t_name}_MeanArea']   = round(t_mean, 2)
            rec[f'Bldg_{t_name}_Proportion'] = t_prop
    else:
        rec['Total_Bldg_Count']      = 0
        rec['Total_Bldg_Area']       = 0.0
        rec['Mean_Bldg_Area']        = 0.0
        rec['Total_Bldg_Proportion'] = 0.0
        rec['Bldg_Density_per_km2']  = 0.0
        for t_name in BLDG_TYPES:
            rec[f'Bldg_{t_name}_Count']      = 0
            rec[f'Bldg_{t_name}_TotalArea']  = 0.0
            rec[f'Bldg_{t_name}_MeanArea']   = 0.0
            rec[f'Bldg_{t_name}_Proportion'] = 0.0

    # ── POIs (extended) ────────────────────────────────────────────
    possible_pois = gdf_pois.iloc[
        list(poi_sindex.intersection(buffer.bounds))]
    precise_pois  = possible_pois[
        possible_pois.intersects(buffer)]

    rec['Total_POI_Count'] = len(precise_pois)

    if len(precise_pois) > 0:
        poi_counts = precise_pois['fclass'].value_counts()
        for p_name, p_tags in POI_TYPES.items():
            rec[f'POI_{p_name}_Count'] = int(
                sum(poi_counts.get(tag, 0) for tag in p_tags))
    else:
        for p_name in POI_TYPES:
            rec[f'POI_{p_name}_Count'] = 0

    # ── LANDUSE (optional) ─────────────────────────────────────────
    if HAS_LANDUSE:
        possible_lu = gdf_landuse.iloc[
            list(lu_sindex.intersection(buffer.bounds))]
        precise_lu  = possible_lu[
            possible_lu.intersects(buffer)].copy()

        if len(precise_lu) > 0:
            precise_lu['clip_area'] = (
                precise_lu.geometry.intersection(buffer).area)
            for lu_col, lu_tags in LU_TYPES.items():
                area = precise_lu[
                    precise_lu['fclass'].isin(lu_tags)
                ]['clip_area'].sum()
                rec[lu_col] = round(float(area), 2)
        else:
            for lu_col in LU_TYPES:
                rec[lu_col] = 0.0
    else:
        for lu_col in LU_TYPES:
            rec[lu_col] = 0.0

    results.append(rec)

    # ── Progress + checkpoint ──────────────────────────────────────
    n_done = loop_idx + 1
    if n_done % 50 == 0 or n_done == len(gdf_dhs):
        print(f'  [{datetime.now().strftime("%H:%M:%S")}] '
              f'{n_done}/{len(gdf_dhs)} clusters done')
    if n_done % 200 == 0:
        cp = output_csv.replace('.csv',
                                f'_checkpoint_{n_done}.csv')
        pd.DataFrame(results).to_csv(cp, index=False)
        print(f'  Checkpoint saved: {cp}')

Extracting features (Roads, Buildings, POIs, Landuse)...
  [20:55:18] 50/1247 clusters done
  [20:55:22] 100/1247 clusters done
  [20:55:24] 150/1247 clusters done
  [20:55:26] 200/1247 clusters done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_expanded_checkpoint_200.csv
  [20:55:27] 250/1247 clusters done
  [20:55:31] 300/1247 clusters done
  [20:55:48] 350/1247 clusters done
  [20:56:04] 400/1247 clusters done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_expanded_checkpoint_400.csv
  [20:56:10] 450/1247 clusters done
  [20:56:12] 500/1247 clusters done
  [20:56:17] 550/1247 clusters done
  [20:56:23] 600/1247 clusters done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_expanded_checkpoint_600.csv
  [20:56:32] 650/1247 clusters done
  [20:56:36] 700/1247 clusters done
  [20:56:38] 750/1247 clusters done
  [20:56:40] 800/1247 clusters done
  Checkpoint saved: /Users

In [7]:
# ==========================================
# 6. MERGE VIIRS + SAVE FULL DATASET
# ==========================================
import time
from datetime import datetime

df_osm = pd.DataFrame(results)
df_osm['DHSCLUST'] = df_osm['DHSCLUST'].astype(int)

print(f'OSM features extracted: {len(df_osm)} clusters')
print(f'Columns: {len(df_osm.columns)}')

# ── Load VIIRS ────────────────────────────────────────────────────
print(f'\nLoading VIIRS data from: {viirs_csv_path}')
df_viirs = pd.read_csv(viirs_csv_path)

col_map = {}
for col in df_viirs.columns:
    if col.lower() in ('cluster_id', 'clusterid'):
        col_map[col] = 'DHSCLUST'
    if col.lower() in ('ntl_value', 'median', 'avg_rad'):
        col_map[col] = 'VIIRS_Median'
if col_map:
    df_viirs = df_viirs.rename(columns=col_map)

if 'DHSCLUST' in df_viirs.columns and 'VIIRS_Median' in df_viirs.columns:
    df_viirs_clean = df_viirs[['DHSCLUST', 'VIIRS_Median']].copy()
elif 'DHSCLUST' in df_viirs.columns and 'NTL_Value' in df_viirs.columns:
    df_viirs_clean = df_viirs[['DHSCLUST', 'NTL_Value']].copy()
    df_viirs_clean = df_viirs_clean.rename(
        columns={'NTL_Value': 'VIIRS_Median'})
else:
    print(f'WARNING: Could not find VIIRS value column.')
    print(f'Available columns: {list(df_viirs.columns)}')
    df_viirs_clean = pd.DataFrame({
        'DHSCLUST'    : df_osm['DHSCLUST'],
        'VIIRS_Median': 0.0
    })

df_viirs_clean['DHSCLUST'] = df_viirs_clean['DHSCLUST'].astype(int)

viirs_ids = set(df_viirs_clean['DHSCLUST'])
osm_ids   = set(df_osm['DHSCLUST'])
missing_viirs = osm_ids - viirs_ids
if missing_viirs:
    print(f'WARNING: {len(missing_viirs)} clusters have no VIIRS data.')
    print(f'  These will get VIIRS_Median = 0 after merge.')

df_final = pd.merge(
    df_osm, df_viirs_clean, on='DHSCLUST', how='left')
df_final['VIIRS_Median'] = df_final['VIIRS_Median'].fillna(0)
df_final = df_final.sort_values(
    'DHSCLUST').reset_index(drop=True)

df_final.to_csv(output_csv, index=False)

# ── Summary ───────────────────────────────────────────────────────
all_cols = list(df_final.columns)

road_cols  = [c for c in all_cols if 'Road' in c or 'Track' in c]
bldg_cols  = [c for c in all_cols if 'Bldg' in c or 'Mean_Bldg' in c
                                   or 'Total_Bldg' in c]
poi_cols   = [c for c in all_cols if c.startswith('POI_')
                                   or c == 'Total_POI_Count']
lu_cols    = [c for c in all_cols if c.startswith('LU_')]
viirs_cols = [c for c in all_cols if 'VIIRS' in c]

print('\n' + '='*60)
print('SUCCESS')
print('='*60)
print(f'Clusters          : {len(df_final)}')
print(f'Total columns     : {len(df_final.columns)}')
print(f'  Road features   : {len(road_cols)}')
print(f'  Building feat.  : {len(bldg_cols)}')
print(f'  POI features    : {len(poi_cols)}')
print(f'  Landuse feat.   : {len(lu_cols)}')
print(f'  VIIRS           : {len(viirs_cols)}')
print(f'VIIRS zeros       : {(df_final["VIIRS_Median"] == 0).sum()}')
print(f'Saved to          : {output_csv}')

print('\n── Road columns ──────────────────────────────────────')
print(road_cols)
print('\n── Building columns ──────────────────────────────────')
print(bldg_cols)
print('\n── POI columns ───────────────────────────────────────')
print(poi_cols)
print('\n── Landuse columns ───────────────────────────────────')
print(lu_cols if lu_cols else '(landuse shapefile not found)')

# Zero-rate audit — flags features with high sparsity
print('\n── Zero rate audit (>60% zeros) ──────────────────────')
feat_cols = [c for c in all_cols if c != 'DHSCLUST']
zero_rates = (df_final[feat_cols] == 0).mean().sort_values(
    ascending=False)
high_zero  = zero_rates[zero_rates > 0.60]
if len(high_zero) > 0:
    print('These features are >60% zero (expected for sparse OSM tags):')
    for col, rate in high_zero.items():
        print(f'  {col}: {rate*100:.1f}%')
else:
    print('No features above 60% zero rate.')

print('\nSample (first 3 rows, first 10 columns):')
print(df_final.iloc[:3, :10].to_string(index=False))

OSM features extracted: 1247 clusters
Columns: 52

Loading VIIRS data from: /Users/ruben/Desktop/Thesis/TrainingData/final-data/viirs_ntl_labels_all_clusters.csv

SUCCESS
Clusters          : 1247
Total columns     : 53
  Road features   : 5
  Building feat.  : 25
  POI features    : 16
  Landuse feat.   : 5
  VIIRS           : 1
VIIRS zeros       : 0
Saved to          : /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_expanded.csv

── Road columns ──────────────────────────────────────
['Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length']

── Building columns ──────────────────────────────────
['Total_Bldg_Count', 'Total_Bldg_Area', 'Mean_Bldg_Area', 'Total_Bldg_Proportion', 'Bldg_Density_per_km2', 'Bldg_residential_Count', 'Bldg_residential_TotalArea', 'Bldg_residential_MeanArea', 'Bldg_residential_Proportion', 'Bldg_commercial_Count', 'Bldg_commercial_TotalArea', 'Bldg_commercial_MeanArea', 'Bldg_commercial_Propor